**# 03 — Silver Layer
 Reads from Bronze Delta table
Casts all StringType columns to correct types
 Deduplicates using MERGE INTO (idempotent)
Adds load_timestamp metadata**

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, LongType, TimestampType, IntegerType
)
from pyspark.sql import functions as F
from datetime import datetime

# ── PARAMETERS ─────────────────────────────
dbutils.widgets.text("load_type", "full", "Load Type (full/incremental)")
dbutils.widgets.text("batch_id", "batch_fullload_2000_2026", "Batch ID")

LOAD_TYPE = dbutils.widgets.get("load_type")
BATCH_ID  = dbutils.widgets.get("batch_id")

print(f"Load Type : {LOAD_TYPE}")
print(f"Batch ID  : {BATCH_ID}")
print("Imports done!")

In [0]:

# READ FROM BRONZE TABLE
# READ FROM BRONZE TABLE (FILTERED BY BATCH)
start_time = datetime.now()

df_bronze = spark.table("pakistan_weather.bronze_weather_raw") \
    .filter(F.col("batch_id") == BATCH_ID)

print(f"Bronze batch rows loaded: {df_bronze.count():,}")
print(f"Bronze columns: {len(df_bronze.columns)}")

In [0]:
df_exploded = df_bronze.select(
    "latitude",
    "longitude",
    "timezone",
    "source_file",
    F.posexplode("hourly.time").alias("idx", "time_str"),
    *[F.col(f"hourly.{v}")[F.col("idx")].alias(v) for v in [
        "temperature_2m", "relative_humidity_2m", "dew_point_2m",
        "apparent_temperature", "pressure_msl", "surface_pressure",
        "weather_code", "vapour_pressure_deficit",
        "precipitation", "rain", "snowfall", "snow_depth",
        "wind_speed_10m", "wind_speed_100m", "wind_direction_10m",
        "wind_direction_100m", "wind_gusts_10m",
        "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
        "shortwave_radiation", "direct_radiation", "diffuse_radiation",
        "direct_normal_irradiance", "global_tilted_irradiance", "terrestrial_radiation",
        "shortwave_radiation_instant", "direct_radiation_instant",
        "diffuse_radiation_instant", "direct_normal_irradiance_instant",
        "global_tilted_irradiance_instant", "terrestrial_radiation_instant",
        "soil_temperature_0_to_7cm", "soil_temperature_7_to_28cm",
        "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
        "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm",
        "soil_moisture_28_to_100cm", "soil_moisture_100_to_255cm",
        "et0_fao_evapotranspiration"
    ]]
).drop("idx")


print(f"Total rows after explode: {df_exploded.count():,}")

In [0]:

# CAST ALL COLUMNS TO CORRECT TYPES

df_silver = df_exploded \
    .withColumn("observation_timestamp",             F.to_timestamp(F.col("time_str"), "yyyy-MM-dd'T'HH:mm")) \
    .withColumn("year",                              F.year(F.col("observation_timestamp"))) \
    .withColumn("month",                             F.month(F.col("observation_timestamp"))) \
    .withColumn("day",                               F.dayofmonth(F.col("observation_timestamp"))) \
    .withColumn("hour",                              F.hour(F.col("observation_timestamp"))) \
    .withColumn("season",                            F.when(F.col("month").isin(3, 4, 5),   "Spring")
                                                      .when(F.col("month").isin(6, 7, 8),   "Summer")
                                                      .when(F.col("month").isin(9, 10, 11), "Autumn")
                                                      .otherwise("Winter")) \
    .withColumn("latitude",                          F.round(F.col("latitude").cast(DoubleType()), 4)) \
    .withColumn("longitude",                         F.round(F.col("longitude").cast(DoubleType()), 4)) \
    .withColumn("temperature_2m",                    F.col("temperature_2m").cast(DoubleType())) \
    .withColumn("relative_humidity_2m",              F.col("relative_humidity_2m").cast(LongType())) \
    .withColumn("dew_point_2m",                      F.col("dew_point_2m").cast(DoubleType())) \
    .withColumn("apparent_temperature",              F.col("apparent_temperature").cast(DoubleType())) \
    .withColumn("pressure_msl",                      F.col("pressure_msl").cast(DoubleType())) \
    .withColumn("surface_pressure",                  F.col("surface_pressure").cast(DoubleType())) \
    .withColumn("weather_code",                      F.col("weather_code").cast(LongType())) \
    .withColumn("vapour_pressure_deficit",           F.col("vapour_pressure_deficit").cast(DoubleType())) \
    .withColumn("precipitation",                     F.col("precipitation").cast(DoubleType())) \
    .withColumn("rain",                              F.col("rain").cast(DoubleType())) \
    .withColumn("snowfall",                          F.col("snowfall").cast(DoubleType())) \
    .withColumn("snow_depth",                        F.col("snow_depth").cast(DoubleType())) \
    .withColumn("wind_speed_10m",                    F.col("wind_speed_10m").cast(DoubleType())) \
    .withColumn("wind_speed_100m",                   F.col("wind_speed_100m").cast(DoubleType())) \
    .withColumn("wind_direction_10m",                F.col("wind_direction_10m").cast(LongType())) \
    .withColumn("wind_direction_100m",               F.col("wind_direction_100m").cast(LongType())) \
    .withColumn("wind_gusts_10m",                    F.col("wind_gusts_10m").cast(DoubleType())) \
    .withColumn("cloud_cover",                       F.col("cloud_cover").cast(LongType())) \
    .withColumn("cloud_cover_low",                   F.col("cloud_cover_low").cast(LongType())) \
    .withColumn("cloud_cover_mid",                   F.col("cloud_cover_mid").cast(LongType())) \
    .withColumn("cloud_cover_high",                  F.col("cloud_cover_high").cast(LongType())) \
    .withColumn("shortwave_radiation",               F.col("shortwave_radiation").cast(DoubleType())) \
    .withColumn("direct_radiation",                  F.col("direct_radiation").cast(DoubleType())) \
    .withColumn("diffuse_radiation",                 F.col("diffuse_radiation").cast(DoubleType())) \
    .withColumn("direct_normal_irradiance",          F.col("direct_normal_irradiance").cast(DoubleType())) \
    .withColumn("global_tilted_irradiance",          F.col("global_tilted_irradiance").cast(DoubleType())) \
    .withColumn("terrestrial_radiation",             F.col("terrestrial_radiation").cast(DoubleType())) \
    .withColumn("shortwave_radiation_instant",       F.col("shortwave_radiation_instant").cast(DoubleType())) \
    .withColumn("direct_radiation_instant",          F.col("direct_radiation_instant").cast(DoubleType())) \
    .withColumn("diffuse_radiation_instant",         F.col("diffuse_radiation_instant").cast(DoubleType())) \
    .withColumn("direct_normal_irradiance_instant",  F.col("direct_normal_irradiance_instant").cast(DoubleType())) \
    .withColumn("global_tilted_irradiance_instant",  F.col("global_tilted_irradiance_instant").cast(DoubleType())) \
    .withColumn("terrestrial_radiation_instant",     F.col("terrestrial_radiation_instant").cast(DoubleType())) \
    .withColumn("soil_temperature_0_to_7cm",         F.col("soil_temperature_0_to_7cm").cast(DoubleType())) \
    .withColumn("soil_temperature_7_to_28cm",        F.col("soil_temperature_7_to_28cm").cast(DoubleType())) \
    .withColumn("soil_temperature_28_to_100cm",      F.col("soil_temperature_28_to_100cm").cast(DoubleType())) \
    .withColumn("soil_temperature_100_to_255cm",     F.col("soil_temperature_100_to_255cm").cast(DoubleType())) \
    .withColumn("soil_moisture_0_to_7cm",            F.col("soil_moisture_0_to_7cm").cast(DoubleType())) \
    .withColumn("soil_moisture_7_to_28cm",           F.col("soil_moisture_7_to_28cm").cast(DoubleType())) \
    .withColumn("soil_moisture_28_to_100cm",         F.col("soil_moisture_28_to_100cm").cast(DoubleType())) \
    .withColumn("soil_moisture_100_to_255cm",        F.col("soil_moisture_100_to_255cm").cast(DoubleType())) \
    .withColumn("et0_fao_evapotranspiration",        F.col("et0_fao_evapotranspiration").cast(DoubleType())) \
    .withColumn("silver_load_timestamp",             F.current_timestamp()) \
    .withColumn("load_type",                         F.lit(LOAD_TYPE)) \
    .withColumn("batch_id",                          F.lit(BATCH_ID)) \
    .withColumn("source_file",                       F.col("source_file")) \
    .drop("time_str")

print(f"Casting done!")
print(f"Silver columns: {len(df_silver.columns)}")

In [0]:
import pyspark.sql.functions as F

# 1. Separate good rows from bad rows
df_good = df_silver.filter(F.col("observation_timestamp").isNotNull())
df_bad  = df_silver.filter(F.col("observation_timestamp").isNull())

if not df_bad.isEmpty():
    df_bad.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("pakistan_weather.silver_quarantine")
    print("Bad rows detected and written to silver_quarantine!")
else:
    print("No bad rows — all data is clean!")

# 3. DEDUPLICATE SOURCE ROWS before merging
# Drop duplicates on the merge primary key columns
merge_keys = ["observation_timestamp", "latitude", "longitude"]
df_good_deduped = df_good.dropDuplicates(merge_keys)

# 4. Create temp view using the deduplicated DataFrame
df_good_deduped.createOrReplaceTempView("silver_updates")

print(f"Original clean rows : {df_good.count():,}")
print(f"Deduplicated rows   : {df_good_deduped.count():,}")
print("Temp view 'silver_updates' created successfully!")

In [0]:

# CREATE SILVER TABLE WITH SCHEMA DRIFT HANDLING
# If table doesn't exist -> create from source schema
# If table exists -> auto-evolve: add any new columns from source via ALTER TABLE
# Also handle columns that exist in target but not in source (add as null to source)

import pyspark.sql.functions as F

TARGET_TABLE = "pakistan_weather.silver_weather_cleaned"
source_cols = df_good_deduped.columns
table_exists = spark.catalog.tableExists(TARGET_TABLE)

if not table_exists:
    # First run: create table from source schema
    spark.sql(f"""
        CREATE TABLE {TARGET_TABLE}
        USING DELTA
        AS SELECT * FROM silver_updates WHERE 1=0
    """)
    print("Silver table created with correct schema!")
else:
    # Table exists — detect and handle schema drift
    target_cols = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]

    # 1. New columns in source that target doesn't have -> ALTER TABLE ADD COLUMNS
    new_cols = [c for c in source_cols if c not in target_cols]
    if new_cols:
        add_cols_sql = ", ".join([f"`{c}` STRING" for c in new_cols])
        spark.sql(f"ALTER TABLE {TARGET_TABLE} ADD COLUMNS ({add_cols_sql})")
        print(f"Schema drift detected! Added {len(new_cols)} new column(s): {new_cols}")
    else:
        print("No schema drift — all source columns match target table.")

    # 2. Columns in target but NOT in source -> add as null to source so MERGE INSERT * works
    target_cols_after = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]
    missing_from_source = [c for c in target_cols_after if c not in source_cols]
    if missing_from_source:
        for c in missing_from_source:
            df_good_deduped = df_good_deduped.withColumn(c, F.lit(None))
        df_good_deduped.createOrReplaceTempView("silver_updates")
        print(f"Added {len(missing_from_source)} null column(s) from target not in source: {missing_from_source}")

spark.sql(f"DESCRIBE {TARGET_TABLE}").show(100, truncate=False)

In [0]:

# MERGE INTO — IDEMPOTENT UPSERT

silver_status = "Success"
silver_error  = None
total_rows = 0

try:
    spark.sql("""
        MERGE INTO pakistan_weather.silver_weather_cleaned AS target
        USING silver_updates AS source
        ON  target.observation_timestamp = source.observation_timestamp
        AND target.latitude               = source.latitude
        AND target.longitude              = source.longitude
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    end_time = datetime.now()

    # Count final rows
    total_rows = spark.sql("""
        SELECT COUNT(*) as cnt 
        FROM pakistan_weather.silver_weather_cleaned
    """).collect()[0]["cnt"]

    print(f"MERGE INTO complete!")
    print(f"Total rows in Silver: {total_rows:,}")
except Exception as e:
    end_time = datetime.now()
    silver_status = "Failure"
    silver_error  = str(e)
    print(f"SILVER MERGE FAILED: {e}")

In [0]:
# ─────────────────────────────────────────────
# LOG THIS SILVER RUN USING DELTA HISTORY METRICS
# ─────────────────────────────────────────────
from delta.tables import DeltaTable

rows_inserted = 0
rows_updated  = 0

if silver_status == "Success":
    # 1. Get exact merge metrics from Delta history
    silver_delta = DeltaTable.forName(spark, "pakistan_weather.silver_weather_cleaned")
    latest_operation = silver_delta.history(1).collect()[0]
    metrics = latest_operation["operationMetrics"]

    rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
    rows_updated  = int(metrics.get("numTargetRowsUpdated", 0))

    # 2. Total rows currently in Silver
    total_rows = spark.sql("""
        SELECT COUNT(*) as cnt
        FROM pakistan_weather.silver_weather_cleaned
    """).collect()[0]["cnt"]

# 3. Insert record into execution logs
log_schema = StructType([
    StructField("layer",          StringType(),    True),
    StructField("load_type",      StringType(),    True),
    StructField("file_processed", StringType(),    True),
    StructField("city",           StringType(),    True),
    StructField("start_time",     TimestampType(), True),
    StructField("end_time",       TimestampType(), True),
    StructField("status",         StringType(),    True),
    StructField("rows_inserted",  LongType(),      True),
    StructField("rows_updated",   LongType(),      True),
    StructField("error_message",  StringType(),    True),
    StructField("batch_id",       StringType(),    True),
])

log_data = [{
    "layer":          "Bronze-to-Silver",
    "load_type":      LOAD_TYPE,
    "file_processed": "pakistan_weather.bronze_weather_raw",
    "city":           "All-5-Cities",
    "start_time":     start_time,
    "end_time":       end_time,
    "status":         silver_status,
    "rows_inserted":  rows_inserted,
    "rows_updated":   rows_updated,
    "error_message":  silver_error,
    "batch_id":       BATCH_ID
}]

log_df = spark.createDataFrame(log_data, schema=log_schema)
log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pakistan_weather.pipeline_execution_logs")

print(f"Rows inserted this batch : {rows_inserted:,}")
print(f"Rows updated this batch  : {rows_updated:,}")
print(f"Total Silver rows        : {total_rows:,}")
print(f"\n{'='*45}")
print(f"SILVER COMPLETE!")
print(f"  Status  : {silver_status}")
print(f"  Duration: {(end_time-start_time).seconds}s")
print(f"{'='*45}")

In [0]:
# ─────────────────────────────────────────────
# VERIFY SILVER
# ─────────────────────────────────────────────
print("Sample Silver data:")
spark.sql("""
    SELECT observation_timestamp, latitude, longitude,
           season, year, month, temperature_2m,
           precipitation, cloud_cover,
           silver_load_timestamp, batch_id
    FROM pakistan_weather.silver_weather_cleaned
    LIMIT 5
""").show(truncate=False)

print("\nRow count:")
spark.sql("""
    SELECT COUNT(*) as total_rows
    FROM pakistan_weather.silver_weather_cleaned
""").show()

print("\nAll pipeline logs:")
spark.sql("""
    SELECT log_id, layer, status, rows_inserted, 
           start_time, end_time, batch_id
    FROM pakistan_weather.pipeline_execution_logs
    ORDER BY log_id
""").show(truncate=False)

In [0]:
print(f"Total rows in Silver: {total_rows:,}")

spark.sql("""
    SELECT count(*)
    FROM pakistan_weather.silver_weather_cleaned
""").show(truncate=False)